In [ ]:
# compliance_fix_rag
 
import os
from dotenv import load_dotenv, find_dotenv
from pyfiglet import figlet_format
from termcolor import colored

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# === Environment Setup ===
load_dotenv(find_dotenv())

# === Constants ===
PDF_DIR = "data_bk/"
VECTOR_DB_PATH = "vectorstore3/"
COMPLIANCE_REPORT = "Compliance_Framework_Report.txt"

# === Stylized Heading ===
def stylish_heading():
    title = figlet_format("ComplianceFixRAG", font="slant", width=100)
    print(colored(title, "cyan"))

# === LLM and Embeddings ===
def load_local_llm():
    return OllamaLLM(model="mistral", temperature=0.1)

def get_embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding_model = get_embedding_model()

# === Step 1: Create Vectorstore from all PDFs ===
def create_pdf_vectorstore():
    docs = []
    for fname in os.listdir(PDF_DIR):
        if fname.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(PDF_DIR, fname))
            docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
    chunks = splitter.split_documents(docs)

    vstore = FAISS.from_documents(chunks, embedding_model)
    os.makedirs(VECTOR_DB_PATH, exist_ok=True)
    vstore.save_local(VECTOR_DB_PATH)
    print("✅ PDF vectorstore created at:", VECTOR_DB_PATH)

# === Prompt Template for Remediation Suggestions ===
SUMMARY_PROMPT = """
You are a compliance expert helping an organization fix their compliance gaps.

Task:
Based on the compliance report provided below, generate a framework-wise summary:
- For each framework (e.g., ISO 27001, GDPR, HIPAA, etc.), identify:
    - Compliant Areas
    - Non-Compliant or Missing Areas
    - Suggestions on how to fix the Non-Compliant Areas (with reference to controls if possible)

Ensure the language is formal, actionable, and specific to each framework.

Compliance Report:
{context}
"""

def get_summary_prompt():
    return PromptTemplate(template=SUMMARY_PROMPT, input_variables=["context"])

# === Step 2: Load the Compliance Report ===
def load_compliance_report():
    with open(COMPLIANCE_REPORT, 'r', encoding='utf-8') as f:
        return f.read()

# === Step 3: Summarize Using RAG + PDF Knowledge ===
def summarize_compliance_fixes():
    report_text = load_compliance_report()
    db = FAISS.load_local(VECTOR_DB_PATH, embedding_model, allow_dangerous_deserialization=True)

    qa_chain = RetrievalQA.from_chain_type(
        llm=load_local_llm(),
        chain_type="stuff",
        retriever=db.as_retriever(search_kwargs={"k": 10}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": get_summary_prompt()},
        input_key="query"
    )

    result = qa_chain.invoke({'query': report_text})
    with open("Fix_Summary_Report.txt", "w", encoding="utf-8") as f:
        f.write(result['result'])

    print("\n📄 Fix Summary saved to 'Fix_Summary_Report.txt'")

# === Run the Workflow ===
if __name__ == "__main__":
    stylish_heading()
    # Run only once to build the vectorstore from PDFs
    # create_pdf_vectorstore()

    summarize_compliance_fixes()
